# 05 — One-Class SVM for Fraud Anomaly Detection

This notebook reconstructs the final **One-Class SVM without `step`** experiment.

Methodology:
- uses the preprocessed 79-feature matrices from notebook 02
- removes `num__step`, leaving 78 features
- trains on a reproducible random sample of 30,000 training transactions
- training is unsupervised: `y_train` is not passed to the model
- model: `OneClassSVM(kernel="rbf", gamma="scale", nu=0.05)`
- anomaly score: `-decision_function(X)` so that higher = more anomalous
- thresholds are selected only on validation
- final evaluation uses the untouched test set
- model, sampled train indices, scores, thresholds and metrics are saved


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

RANDOM_STATE = 42
TRAIN_SAMPLE_SIZE = 30_000


## Paths

In [2]:
PROCESSED_PATH = Path("../data/processed")
MODELS_PATH = Path("../models")
RESULTS_PATH = Path("../results")
SCORES_PATH = RESULTS_PATH / "scores"
METRICS_PATH = RESULTS_PATH / "metrics"

MODELS_PATH.mkdir(parents=True, exist_ok=True)
SCORES_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)


## Load preprocessed data

In [3]:
X_train = sparse.load_npz(PROCESSED_PATH / "X_train.npz")
X_valid = sparse.load_npz(PROCESSED_PATH / "X_valid.npz")
X_test = sparse.load_npz(PROCESSED_PATH / "X_test.npz")

y_train = np.load(PROCESSED_PATH / "y_train.npy")
y_valid = np.load(PROCESSED_PATH / "y_valid.npy")
y_test = np.load(PROCESSED_PATH / "y_test.npy")

print("Train:", X_train.shape, y_train.shape)
print("Valid:", X_valid.shape, y_valid.shape)
print("Test :", X_test.shape, y_test.shape)


Train: (374914, 79) (374914,)
Valid: (108423, 79) (108423,)
Test : (111306, 79) (111306,)


## Remove absolute `step` feature

In [4]:
with open(MODELS_PATH / "feature_names.json", "r") as f:
    feature_names = json.load(f)

step_feature = "num__step"
step_index = feature_names.index(step_feature)

keep_indices = [
    i for i in range(len(feature_names))
    if i != step_index
]

ocsvm_feature_names = [
    feature_names[i]
    for i in keep_indices
]

X_train_no_step = X_train[:, keep_indices]
X_valid_no_step = X_valid[:, keep_indices]
X_test_no_step = X_test[:, keep_indices]

print("step index:", step_index)
print("Train:", X_train_no_step.shape)
print("Valid:", X_valid_no_step.shape)
print("Test :", X_test_no_step.shape)


step index: 0
Train: (374914, 78)
Valid: (108423, 78)
Test : (111306, 78)


Expected dimensionality after removing `step`: **78 features**.

This ablation is important because the original RBF One-Class SVM with absolute `step` suffered severe score drift between validation and test.


## Reproducible 30,000-row training sample

In [5]:
rng = np.random.RandomState(RANDOM_STATE)

train_indices = rng.choice(
    X_train_no_step.shape[0],
    size=TRAIN_SAMPLE_SIZE,
    replace=False
)

X_train_sample = X_train_no_step[
    train_indices
]

print("Training sample shape:", X_train_sample.shape)
print("First sampled indices:", train_indices[:10])


Training sample shape: (30000, 78)
First sampled indices: [174143 230327  99810 106493  69217 203288 167589 105828 148966 176154]


The sample is selected **without using `y_train`**.

The indices are saved at the end so this exact training subset can be reused later.


## Build One-Class SVM

In [6]:
ocsvm_no_step = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

ocsvm_no_step


OneClassSVM(nu=0.05)

`nu=0.05` is a One-Class SVM regularization parameter. It is **not** the fraud rate.


## Train

In [7]:
ocsvm_no_step.fit(
    X_train_sample
)

print("One-Class SVM training complete.")


One-Class SVM training complete.


## Compute anomaly scores

Scikit-learn's native `decision_function` gives larger values to observations considered more normal.

We use:

`anomaly_score = -decision_function(X)`

Therefore, higher score = more anomalous.

The score is not a probability.


In [8]:
valid_scores_no_step = -ocsvm_no_step.decision_function(
    X_valid_no_step
)

test_scores_no_step = -ocsvm_no_step.decision_function(
    X_test_no_step
)

print("Validation scores:", valid_scores_no_step.shape)
print("Test scores:", test_scores_no_step.shape)


Validation scores: (108423,)
Test scores: (111306,)


## Score distribution by class

In [9]:
valid_score_analysis = pd.DataFrame({
    "fraud": y_valid,
    "anomaly_score": valid_scores_no_step
})

valid_score_analysis.groupby(
    "fraud"
)["anomaly_score"].describe()


,count,mean,std,min,25%,50%,75%,max
fraud,,,,,,,,
0,107223.0,-9.630827,6.781154,-27.087742,-12.458621,-9.472916,-6.799712,114.590037
1,1200.0,26.441288,37.657246,-15.794646,-0.306423,15.802218,38.335429,171.701215


In [10]:
test_score_analysis = pd.DataFrame({
    "fraud": y_test,
    "anomaly_score": test_scores_no_step
})

test_score_analysis.groupby(
    "fraud"
)["anomaly_score"].describe()


,count,mean,std,min,25%,50%,75%,max
fraud,,,,,,,,
0,110106.0,-9.705471,6.641313,-27.087742,-12.525643,-9.546192,-6.883759,107.307607
1,1200.0,24.446044,36.109128,-15.206525,-0.841307,13.726242,36.624888,172.649457


Historical no-`step` run reference:

- validation score mean ≈ `-8.0516`
- validation median ≈ `-8.0256`
- test score mean ≈ `-8.1115`
- test median ≈ `-8.0439`

The near-identical validation/test distributions showed that removing absolute time solved the previous calibration drift.


## Ranking metrics

In [11]:
valid_roc_auc_ocsvm = roc_auc_score(
    y_valid,
    valid_scores_no_step
)

valid_pr_auc_ocsvm = average_precision_score(
    y_valid,
    valid_scores_no_step
)

test_roc_auc_ocsvm = roc_auc_score(
    y_test,
    test_scores_no_step
)

test_pr_auc_ocsvm = average_precision_score(
    y_test,
    test_scores_no_step
)

print(
    f"Validation ROC-AUC : {valid_roc_auc_ocsvm:.4f}"
)

print(
    f"Validation PR-AUC  : {valid_pr_auc_ocsvm:.4f}"
)

print()

print(
    f"Test ROC-AUC       : {test_roc_auc_ocsvm:.4f}"
)

print(
    f"Test PR-AUC        : {test_pr_auc_ocsvm:.4f}"
)


Validation ROC-AUC : 0.9249
Validation PR-AUC  : 0.4473

Test ROC-AUC       : 0.9255
Test PR-AUC        : 0.4370


Historical no-`step` reference:

- Validation ROC-AUC ≈ **0.9217**
- Validation PR-AUC ≈ **0.4555**
- Test ROC-AUC ≈ **0.9220**
- Test PR-AUC ≈ **0.4414**

Your new run may differ if the original 30,000-row sample was generated differently. Keep the values actually produced by this reproducible run.


## Validation threshold — Max F1

In [12]:
precision_ocsvm, recall_ocsvm, thresholds_ocsvm = (
    precision_recall_curve(
        y_valid,
        valid_scores_no_step
    )
)

f1_scores_ocsvm = (
    2
    * precision_ocsvm[:-1]
    * recall_ocsvm[:-1]
    / (
        precision_ocsvm[:-1]
        + recall_ocsvm[:-1]
        + 1e-12
    )
)

best_index_ocsvm = np.argmax(
    f1_scores_ocsvm
)

best_threshold_ocsvm = thresholds_ocsvm[
    best_index_ocsvm
]

best_precision_ocsvm = precision_ocsvm[
    best_index_ocsvm
]

best_recall_ocsvm = recall_ocsvm[
    best_index_ocsvm
]

best_f1_ocsvm = f1_scores_ocsvm[
    best_index_ocsvm
]

print(
    f"Best threshold : {best_threshold_ocsvm:.6f}"
)
print(
    f"Precision      : {best_precision_ocsvm:.4f}"
)
print(
    f"Recall         : {best_recall_ocsvm:.4f}"
)
print(
    f"F1-score       : {best_f1_ocsvm:.4f}"
)


Best threshold : 23.177817
Precision      : 0.5544
Recall         : 0.4075
F1-score       : 0.4697


Historical validation reference:

- threshold ≈ `25.420965`
- precision ≈ `0.5550`
- recall ≈ `0.3992`
- F1 ≈ `0.4644`


## Validation threshold — High Recall

In [13]:
TARGET_RECALL = 0.80

precision_thresholds_ocsvm = precision_ocsvm[:-1]
recall_thresholds_ocsvm = recall_ocsvm[:-1]

candidate_indices = np.where(
    recall_thresholds_ocsvm >= TARGET_RECALL
)[0]

best_recall_index_ocsvm = candidate_indices[
    np.argmax(
        precision_thresholds_ocsvm[
            candidate_indices
        ]
    )
]

recall80_threshold_ocsvm = thresholds_ocsvm[
    best_recall_index_ocsvm
]

recall80_precision_ocsvm = (
    precision_thresholds_ocsvm[
        best_recall_index_ocsvm
    ]
)

recall80_recall_ocsvm = (
    recall_thresholds_ocsvm[
        best_recall_index_ocsvm
    ]
)

recall80_f1_ocsvm = (
    2
    * recall80_precision_ocsvm
    * recall80_recall_ocsvm
    / (
        recall80_precision_ocsvm
        + recall80_recall_ocsvm
        + 1e-12
    )
)


In [14]:
ocsvm_validation_thresholds = pd.DataFrame({
    "Strategy": [
        "Max F1",
        "Recall >= 80%"
    ],
    "Threshold": [
        best_threshold_ocsvm,
        recall80_threshold_ocsvm
    ],
    "Precision": [
        best_precision_ocsvm,
        recall80_precision_ocsvm
    ],
    "Recall": [
        best_recall_ocsvm,
        recall80_recall_ocsvm
    ],
    "F1": [
        best_f1_ocsvm,
        recall80_f1_ocsvm
    ]
})

ocsvm_validation_thresholds


,Strategy,Threshold,Precision,Recall,F1
0,Max F1,23.177817,0.554422,0.4075,0.469741
1,Recall >= 80%,-2.386012,0.109502,0.8000,0.192636


Historical high-recall validation reference:

- threshold ≈ `-1.759274`
- precision ≈ `0.1240`
- recall ≈ `0.8000`
- F1 ≈ `0.2148`


## Apply frozen validation thresholds to test

In [15]:
y_test_pred_ocsvm_f1 = (
    test_scores_no_step >= best_threshold_ocsvm
).astype(int)

y_test_pred_ocsvm_r80 = (
    test_scores_no_step >= recall80_threshold_ocsvm
).astype(int)


## Evaluation helper

In [16]:
def evaluate_predictions(
    y_true,
    y_pred
):
    precision = precision_score(
        y_true,
        y_pred
    )

    recall = recall_score(
        y_true,
        y_pred
    )

    f1 = f1_score(
        y_true,
        y_pred
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    false_positive_rate = fp / (fp + tn)
    alert_rate = y_pred.mean()

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "False Positive Rate": false_positive_rate,
        "Alert Rate": alert_rate,
        "True Positives": tp,
        "False Positives": fp,
        "False Negatives": fn,
        "True Negatives": tn
    }


## Final test evaluation

In [17]:
results_ocsvm_f1 = evaluate_predictions(
    y_test,
    y_test_pred_ocsvm_f1
)

results_ocsvm_r80 = evaluate_predictions(
    y_test,
    y_test_pred_ocsvm_r80
)


In [18]:
ocsvm_test_results = pd.DataFrame([
    {
        "Strategy": "Max F1",
        "Precision": results_ocsvm_f1["Precision"],
        "Recall": results_ocsvm_f1["Recall"],
        "F1": results_ocsvm_f1["F1"],
        "False Positive Rate":
            results_ocsvm_f1["False Positive Rate"],
        "Alert Rate":
            results_ocsvm_f1["Alert Rate"]
    },
    {
        "Strategy": "Recall >= 80%",
        "Precision": results_ocsvm_r80["Precision"],
        "Recall": results_ocsvm_r80["Recall"],
        "F1": results_ocsvm_r80["F1"],
        "False Positive Rate":
            results_ocsvm_r80["False Positive Rate"],
        "Alert Rate":
            results_ocsvm_r80["Alert Rate"]
    }
])

ocsvm_test_results


,Strategy,Precision,Recall,F1,False Positive Rate,Alert Rate
0,Max F1,0.572500,0.381667,0.458000,0.003106,0.007187
1,Recall >= 80%,0.106891,0.795000,0.188444,0.072394,0.080184


In [19]:
for strategy, results in [
    ("Max F1", results_ocsvm_f1),
    ("Recall >= 80%", results_ocsvm_r80)
]:
    print(f"\n--- {strategy} ---")
    print(
        "Frauds detected :",
        f"{results['True Positives']:,}"
    )
    print(
        "Frauds missed   :",
        f"{results['False Negatives']:,}"
    )
    print(
        "False alerts    :",
        f"{results['False Positives']:,}"
    )
    print(
        "True negatives  :",
        f"{results['True Negatives']:,}"
    )



--- Max F1 ---
Frauds detected : 458
Frauds missed   : 742
False alerts    : 342
True negatives  : 109,764

--- Recall >= 80% ---
Frauds detected : 954
Frauds missed   : 246
False alerts    : 7,971
True negatives  : 102,135


Historical test reference:

### Max F1
- Precision ≈ **0.5602**
- Recall ≈ **0.3800**
- F1 ≈ **0.4528**
- FPR ≈ **0.00325**
- Alert Rate ≈ **0.00731**

### High Recall
- Precision ≈ **0.1187**
- Recall ≈ **0.7850**
- F1 ≈ **0.2062**
- FPR ≈ **0.0635**
- Alert Rate ≈ **0.0713**


## Save trained model

In [20]:
joblib.dump(
    ocsvm_no_step,
    MODELS_PATH / "one_class_svm_no_step.joblib"
)

print("One-Class SVM saved.")


One-Class SVM saved.


## Save exact sampled train indices

In [21]:
np.save(
    MODELS_PATH / "one_class_svm_train_indices.npy",
    train_indices
)

print("Training indices saved.")


Training indices saved.


## Save feature names

In [22]:
with open(
    MODELS_PATH / "one_class_svm_feature_names.json",
    "w"
) as f:
    json.dump(
        ocsvm_feature_names,
        f,
        indent=2
    )


## Save anomaly scores

In [23]:
np.save(
    SCORES_PATH / "ocsvm_valid_scores.npy",
    valid_scores_no_step
)

np.save(
    SCORES_PATH / "ocsvm_test_scores.npy",
    test_scores_no_step
)

print("One-Class SVM scores saved.")


One-Class SVM scores saved.


## Save thresholds

In [24]:
thresholds_to_save = {
    "max_f1": {
        "threshold":
            float(best_threshold_ocsvm),
        "validation_precision":
            float(best_precision_ocsvm),
        "validation_recall":
            float(best_recall_ocsvm),
        "validation_f1":
            float(best_f1_ocsvm)
    },
    "high_recall": {
        "target_recall":
            TARGET_RECALL,
        "threshold":
            float(recall80_threshold_ocsvm),
        "validation_precision":
            float(recall80_precision_ocsvm),
        "validation_recall":
            float(recall80_recall_ocsvm),
        "validation_f1":
            float(recall80_f1_ocsvm)
    }
}

with open(
    MODELS_PATH / "one_class_svm_thresholds.json",
    "w"
) as f:
    json.dump(
        thresholds_to_save,
        f,
        indent=2
    )

print("Thresholds saved.")


Thresholds saved.


## Save metrics

In [25]:
ranking_metrics_ocsvm = pd.DataFrame([
    {
        "Model": "One-Class SVM",
        "Validation ROC-AUC":
            valid_roc_auc_ocsvm,
        "Validation PR-AUC":
            valid_pr_auc_ocsvm,
        "Test ROC-AUC":
            test_roc_auc_ocsvm,
        "Test PR-AUC":
            test_pr_auc_ocsvm
    }
])

ranking_metrics_ocsvm.to_csv(
    METRICS_PATH / "ocsvm_ranking_metrics.csv",
    index=False
)

ocsvm_test_results.to_csv(
    METRICS_PATH / "ocsvm_operational_metrics.csv",
    index=False
)

print("Metrics saved.")


Metrics saved.


## Final verification

In [26]:
print("MODEL")
print(MODELS_PATH / "one_class_svm_no_step.joblib")

print("\nTRAIN INDICES")
print(MODELS_PATH / "one_class_svm_train_indices.npy")

print("\nSCORES")
print(SCORES_PATH / "ocsvm_valid_scores.npy")
print(SCORES_PATH / "ocsvm_test_scores.npy")

print("\nTHRESHOLDS")
print(MODELS_PATH / "one_class_svm_thresholds.json")

print("\nMETRICS")
print(METRICS_PATH / "ocsvm_ranking_metrics.csv")
print(METRICS_PATH / "ocsvm_operational_metrics.csv")

print("\nDone.")


MODEL
..\models\one_class_svm_no_step.joblib

TRAIN INDICES
..\models\one_class_svm_train_indices.npy

SCORES
..\results\scores\ocsvm_valid_scores.npy
..\results\scores\ocsvm_test_scores.npy

THRESHOLDS
..\models\one_class_svm_thresholds.json

METRICS
..\results\metrics\ocsvm_ranking_metrics.csv
..\results\metrics\ocsvm_operational_metrics.csv

Done.


## Conclusion

The final One-Class SVM experiment deliberately excludes absolute `step`.

The earlier version with `step` suffered severe future score drift because the RBF kernel treated later time periods as progressively farther from the training support. Removing `step` stabilized the validation/test score distributions and substantially improved operational threshold transfer.

From this point onward, all artifacts required for exact reuse are persisted to disk.
